In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score
)

RANDOM_STATE = 42

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)
print("Files:", os.listdir(path))


## 2. Load the dataset from the KaggleHub path

In [ ]:
csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError(f"No CSV file found in: {path}")

csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

print("Dataset file:", csv_path)
print("Original shape:", df.shape)
display(df.head())


## 3. Create the required 20,000-row sample

The dataset is highly imbalanced. We keep **all fraud transactions** and randomly sample legitimate transactions until the dataset contains exactly 20,000 rows.


In [ ]:
fraud_df = df[df["Class"] == 1].copy()
legitimate_df = df[df["Class"] == 0].copy()

n_legitimate_needed = 20000 - len(fraud_df)

if n_legitimate_needed <= 0:
    raise ValueError("The number of fraud rows is already >= 20,000.")

legitimate_sample = legitimate_df.sample(
    n=n_legitimate_needed,
    random_state=RANDOM_STATE
)

sampled_df = pd.concat([fraud_df, legitimate_sample], axis=0)
sampled_df = sampled_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Sampled shape:", sampled_df.shape)

class_counts = sampled_df["Class"].value_counts().sort_index()
class_percentages = sampled_df["Class"].value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "Class": ["Legitimate (0)", "Fraud (1)"],
    "Count": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "Percentage": [class_percentages.get(0, 0), class_percentages.get(1, 0)]
})

display(class_summary.round(2))


## 4. Task 1 — Stratified 80/20 train/test split

In [ ]:
X = sampled_df.drop(columns=["Class"])
y = sampled_df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training fraud percentage:", round(y_train.mean() * 100, 2), "%")
print("Testing fraud percentage:", round(y_test.mean() * 100, 2), "%")


## 5. Task 1 — Baseline Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train, y_train)

y_pred_lr = logistic_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression Test Accuracy: {lr_accuracy:.4f}")


## 6. Task 2 — Confusion Matrix

For the fraud class:
- **TP:** Fraud correctly detected
- **FP:** Legitimate transaction incorrectly flagged as fraud
- **TN:** Legitimate transaction correctly classified
- **FN:** Fraud transaction missed


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel()

print("Confusion Matrix:")
display(pd.DataFrame(
    cm_lr,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
))

print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")


## 7. Task 2 — Precision, Recall, and F1 for the Fraud Class

In [ ]:
lr_precision = precision_score(y_test, y_pred_lr, pos_label=1)
lr_recall = recall_score(y_test, y_pred_lr, pos_label=1)
lr_f1 = f1_score(y_test, y_pred_lr, pos_label=1)

print(f"Fraud Precision: {lr_precision:.4f}")
print(f"Fraud Recall:    {lr_recall:.4f}")
print(f"Fraud F1-score:  {lr_f1:.4f}")


### Why accuracy alone is misleading

Fraud is a small minority of the transactions, so a model can achieve very high accuracy mainly by correctly classifying legitimate transactions. In fraud detection, **False Negatives are especially important**, because they represent real fraudulent transactions that were missed; therefore, fraud-class **Recall and F1-score** provide a much better view of useful model performance than accuracy alone.


## 8. Task 3 — Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, pos_label=1)
rf_recall = recall_score(y_test, y_pred_rf, pos_label=1)
rf_f1 = f1_score(y_test, y_pred_rf, pos_label=1)

print(f"Random Forest Test Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest Fraud Precision: {rf_precision:.4f}")
print(f"Random Forest Fraud Recall:    {rf_recall:.4f}")
print(f"Random Forest Fraud F1-score:  {rf_f1:.4f}")


## 9. 5-Fold Cross-Validation — F1-score

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

lr_cv_scores = cross_val_score(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

rf_cv_scores = cross_val_score(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Logistic Regression F1": lr_cv_scores,
    "Random Forest F1": rf_cv_scores
})

display(cv_results.round(4))

print(f"Mean Logistic Regression CV F1: {lr_cv_scores.mean():.4f}")
print(f"Mean Random Forest CV F1:       {rf_cv_scores.mean():.4f}")


## 10. ROC Curve and AUC

In [ ]:
lr_prob = logistic_model.predict_proba(X_test)[:, 1]
rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)

lr_auc = roc_auc_score(y_test, lr_prob)
rf_auc = roc_auc_score(y_test, rf_prob)

plt.figure(figsize=(9, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {lr_auc:.4f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — SwiftCart Fraud Detection")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Random Forest AUC:       {rf_auc:.4f}")


## 11. Final Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Test Accuracy",
        "Fraud Precision",
        "Fraud Recall",
        "Fraud F1-score",
        "Mean 5-Fold CV F1",
        "AUC"
    ],
    "Logistic Regression": [
        lr_accuracy, lr_precision, lr_recall, lr_f1,
        lr_cv_scores.mean(), lr_auc
    ],
    "Random Forest": [
        rf_accuracy, rf_precision, rf_recall, rf_f1,
        rf_cv_scores.mean(), rf_auc
    ]
})

display(comparison.round(4))


## 12. Production Recommendation

**Recommendation: Deploy the Random Forest model.** The final decision should prioritize fraud-detection performance rather than accuracy alone, especially the fraud-class F1-score and recall. Random Forest should be selected if it provides the stronger F1 performance in the executed results, while AUC is also considered as a measure of ranking/discrimination quality. Although Logistic Regression is simpler and may achieve a competitive AUC, Random Forest is the preferred production candidate when its fraud F1 and cross-validated F1 are higher, because that indicates a better balance between detecting fraud and limiting false alarms.


### Final takeaway

The analysis demonstrates why SwiftCart should not evaluate fraud models using accuracy alone. The combination of **fraud Precision, Recall, F1-score, 5-fold cross-validation F1, and ROC-AUC** provides a more appropriate assessment for this highly imbalanced classification problem.
